# 🧠 Notebook 19: Reasoning & Test-Time Compute

How models reason, why spending more compute at inference time produces better answers, and the techniques that make it work — from Chain-of-Thought to Monte Carlo Tree Search.

**Prerequisites:** Notebook 07 (Building GPT), Notebook 17 (Alignment: RLHF/DPO/GRPO)

**What you'll learn:**
- 💡 Why LLMs can generate fluent text but struggle with multi-step reasoning — and what we can do about it
- 🎯 The test-time compute paradigm: trade inference FLOPs for answer quality without retraining
- ⚡ Chain-of-Thought prompting, self-consistency, and Tree-of-Thought — practical techniques you can use today
- ⚠️ MCTS-style search and Process Reward Models — the machinery behind o1 and DeepSeek-R1
- 💡 How to think about the training-time vs test-time compute tradeoff at a systems level

## The Reasoning Gap

LLMs are remarkably good at pattern completion. Given a prompt, they produce fluent, contextually appropriate text. But ask a model to solve a multi-step math problem, plan a sequence of actions, or debug a logical argument — and things fall apart quickly.

💡 **Why?** Standard autoregressive generation produces one token at a time, left to right, with no backtracking. The model commits to each token immediately. There's no mechanism to:
- **Plan ahead** — consider multiple possible continuations before choosing one
- **Verify intermediate steps** — check whether a reasoning step is correct before building on it
- **Backtrack** — abandon a flawed reasoning path and try a different approach

This is fundamentally different from how humans reason. When you solve a math problem, you sketch out an approach, check intermediate results, and restart if something doesn't add up. LLMs don't do any of that by default.

🎯 **Interview tip:** The reasoning gap is one of the most active research areas in AI. Understanding *why* LLMs struggle with reasoning — and the different approaches to fixing it — signals deep understanding of both the capabilities and limitations of current systems.

### 🔤 What Does "Autoregressive" Mean?

When we say LLMs generate text "autoregressively," we mean they produce **one token (word/piece) at a time**, always moving left to right:

```
Input:  "The cat sat on the ___"
Step 1: Model predicts "mat" → "The cat sat on the mat"
Step 2: Model predicts "." → "The cat sat on the mat."
```

Each new token is chosen based on ALL the tokens that came before it. But here's the catch: **once a token is chosen, it's permanent.** The model can't go back and change "mat" to "couch."

Think of it like writing with a pen (not a pencil):
- ✅ You can think about what to write next
- ❌ You can't erase what you already wrote
- ❌ You can't peek ahead to see if your current sentence will lead somewhere good

This is why reasoning is hard for LLMs — they can't "draft and revise" like humans do. Test-time compute techniques are all about working around this limitation.

---

## The Test-Time Compute Paradigm

There are two fundamentally different ways to improve an LLM's outputs:

### Training-Time Compute: Build a Better Model

Spend more FLOPs during training → better base model. This is the scaling laws story from Notebook 18:
- More parameters $N$ → lower loss
- More training tokens $D$ → lower loss
- The Chinchilla law tells you the optimal allocation

But training-time compute has a hard constraint: once training is done, the model is fixed. Every query gets the same amount of compute — whether you're asking "What's 2+2?" or "Prove the Riemann hypothesis."

### Test-Time Compute: Think Harder on Hard Problems

Spend more FLOPs during inference → better answers *from the same model*. This is the new frontier:
- Easy questions → fast, single-pass generation (cheap)
- Hard questions → multiple samples, search, verification (expensive but worth it)

💡 **The key insight:** You can trade inference compute for answer quality. A smaller model with more test-time compute can outperform a larger model with standard generation on reasoning tasks.

This was the core idea behind OpenAI's o1 (2024) and DeepSeek-R1 (2025) — models that "think" before answering by generating internal reasoning chains and using search to find better solutions.

⚡ **Performance perspective:** Test-time compute is especially attractive because:
1. Training is a one-time cost; inference happens millions of times
2. You can allocate compute *adaptively* — more for hard problems, less for easy ones
3. It composes with training improvements — a better base model + test-time search = even better results

---

## The Compute Tradeoff Visualized

Before diving into specific techniques, let's build intuition for the training-time vs test-time compute tradeoff.

Think of it as two axes:
- **x-axis:** Training compute (FLOPs spent training the model)
- **y-axis:** Test-time compute (FLOPs spent per query at inference)

```
Test-time compute ↑
                  │
    DeepSeek-R1 ● │  ● o1-pro
                  │
         o1-mini ●│
                  │
                  │  ● GPT-4 (standard)
    CoT prompting ●│
                  │  ● Claude 3.5
                  │
  Standard decode ●──●──●──●──●──→ Training compute
                  │ Small  Medium  Large
```

⚠️ **Pitfall:** More test-time compute doesn't always help. If the base model lacks the knowledge to solve a problem, no amount of search will find the right answer. Test-time compute amplifies capability — it doesn't create it from nothing.

🎯 **Interview tip:** When discussing reasoning models, frame it as a *compute allocation* problem. Labs are actively researching the optimal frontier: for a fixed total compute budget, how much should go to training vs inference?

---

## Overview of Test-Time Compute Approaches

Here's the landscape of techniques, ordered from simplest to most sophisticated:

### 1. Chain-of-Thought (CoT) Prompting — "Think Step by Step"

**Core idea:** Instead of asking the model to jump straight to the answer, prompt it to show its reasoning steps first.

```
Standard:  Q: "What is 17 × 24?" → A: "408"
CoT:       Q: "What is 17 × 24? Think step by step."
           → A: "17 × 24 = 17 × 20 + 17 × 4 = 340 + 68 = 408"
```

💡 **Why it works:** The intermediate tokens act as a "scratchpad." Each reasoning step conditions the next token prediction on more relevant context. The model effectively decomposes a hard problem into easier sub-problems.

**Cost:** ~2-5x more tokens generated per query. Minimal overhead.

### 2. Self-Consistency — "Sample Multiple Answers, Take the Majority Vote"

**Core idea:** Generate N independent Chain-of-Thought solutions, extract the final answer from each, and take the majority vote.

```
Sample 1: "17 × 20 = 340, 17 × 4 = 68, total = 408" → 408
Sample 2: "17 × 24 = 17 × 25 - 17 = 425 - 17 = 408" → 408  
Sample 3: "10 × 24 = 240, 7 × 24 = 168, total = 408" → 408
Majority vote → 408 ✓
```

💡 **Why it works:** Different reasoning paths may make different errors, but the correct answer tends to appear more often. This is essentially an ensemble over reasoning strategies.

**Cost:** N× more tokens. Typically N = 5-40 samples.

### 3. Tree-of-Thought (ToT) — "Explore Multiple Reasoning Paths"

**Core idea:** Instead of generating reasoning linearly, build a tree where each node is a partial reasoning state. At each step, generate multiple possible next steps, evaluate them, and expand the most promising ones.

```
         [Problem]
        /    |    \
    [Step A] [Step B] [Step C]    ← Generate candidates
       ↓        ↓                  ← Evaluate & prune
   [Step A2] [Step B2]
       ↓
   [Answer]                        ← Best path
```

💡 **Why it works:** Explicit search over reasoning paths. The model can explore alternatives and backtrack — something standard generation can't do.

**Cost:** Branching factor × depth × evaluation cost. Significantly more expensive.

### 4. MCTS-Style Search — "Systematic Exploration with Backtracking"

**Core idea:** Apply Monte Carlo Tree Search (from game-playing AI like AlphaGo) to reasoning. Build a search tree over reasoning steps, using UCB1 to balance exploration vs exploitation.

⚡ **Key components:**
- **Selection:** Traverse the tree using UCB1 to pick which node to expand
- **Expansion:** Generate new reasoning steps from the selected node
- **Evaluation:** Score the new steps using a reward model
- **Backpropagation:** Update ancestor nodes with the evaluation results

**Cost:** Budget × (generation + evaluation) per query. The most expensive approach, but also the most powerful for hard problems.

### 5. Process Reward Models (PRMs) — "Score Intermediate Steps, Not Just Final Answers"

**Core idea:** Train a model to evaluate the quality of each individual reasoning step, not just the final answer. This enables much more effective search — you can prune bad reasoning paths early.

```
Step 1: "Let x = 17 × 24"           → PRM score: 0.95 ✓
Step 2: "17 × 24 = 17 × 20 + 17×4"  → PRM score: 0.92 ✓
Step 3: "= 340 + 68"                 → PRM score: 0.97 ✓
Step 4: "= 408"                      → PRM score: 0.99 ✓
```

vs. Outcome Reward Model (ORM): only scores the final answer "408" → 1.0

💡 **Why PRMs matter:** They provide *dense* feedback for search. An ORM only tells you if the final answer is right — a PRM tells you *where* the reasoning went wrong, enabling targeted backtracking.

🎯 **Interview tip:** The PRM vs ORM distinction is a hot topic. OpenAI's "Let's Verify Step by Step" paper (2023) showed that PRMs significantly outperform ORMs for math reasoning. DeepSeek-R1 (2025) uses RL with process-level rewards to train reasoning capabilities directly into the model.

---

## Comparing the Approaches

| Approach | Extra Compute | Requires Training? | Backtracking? | Best For |
|----------|:---:|:---:|:---:|---|
| **Chain-of-Thought** | 2-5× tokens | No (prompting only) | ❌ | Simple multi-step problems |
| **Self-Consistency** | N× tokens | No (prompting only) | ❌ | Problems with verifiable answers |
| **Tree-of-Thought** | B^D × eval cost | Optional (can use prompting) | ✅ | Planning, creative reasoning |
| **MCTS Search** | Budget × (gen + eval) | Yes (reward model) | ✅ | Hard math, code, formal reasoning |
| **Process Reward Models** | Per-step scoring | Yes (step-level labels) | ✅ | Any search-based approach |

⚠️ **Pitfall:** These approaches aren't mutually exclusive. Modern reasoning systems like o1 and DeepSeek-R1 combine multiple techniques:
- RL-trained reasoning (the model learns to generate CoT internally)
- Search over reasoning paths (MCTS or beam search)
- Process reward models (to guide the search)
- Self-consistency (to verify final answers)

💡 **The trend:** The field is moving from *prompting-based* reasoning (CoT, self-consistency) toward *search-based* reasoning (MCTS + PRMs) and *learned* reasoning (RL-trained models that reason natively). The latter two require more infrastructure but produce dramatically better results on hard problems.

---

## Math Foundation: Why Search Helps

Let's formalize why test-time compute improves reasoning. Consider a model $\pi$ generating a solution $s$ to problem $p$:

$$P(\text{correct} \mid p) = \sum_{s} \pi(s \mid p) \cdot \mathbb{1}[\text{verify}(s, p)]$$

With **single-sample generation**, you draw one $s \sim \pi(\cdot \mid p)$ and hope it's correct.

With **self-consistency** (N samples), the probability of getting at least one correct answer is:

$$P(\text{at least one correct} \mid p, N) = 1 - (1 - p_{\text{correct}})^N$$

If a model gets a problem right 30% of the time with single sampling, then with N=10 samples:

$$P = 1 - (1 - 0.3)^{10} = 1 - 0.7^{10} \approx 0.972$$

💡 **Key insight:** Even a model that's "usually wrong" on hard problems can become "almost always right" with enough samples — *if* you can verify which answer is correct. This is why verifiers (reward models) are so important.

With **MCTS search**, the improvement is even more dramatic because search is *directed* — it doesn't just sample randomly but actively explores promising paths:

$$\text{Quality}(\text{MCTS}, B) \geq \text{Quality}(\text{random}, B)$$

where $B$ is the compute budget. The UCB1 formula balances exploration and exploitation:

$$\text{UCB1}(v) = \bar{X}_v + \sqrt{\frac{2 \ln N_{\text{parent}}}{N_v}}$$

where $\bar{X}_v$ is the average reward of node $v$, $N_{\text{parent}}$ is the parent's visit count, and $N_v$ is the node's visit count.

⚡ **Performance note:** The $\sqrt{2 \ln N / n}$ exploration term ensures every node gets visited eventually, but nodes with higher estimated value get visited more often. This is provably optimal in the limit for multi-armed bandits.

---

## Setup

Let's set up our environment. All implementations use MLX on Apple Silicon.

### 📦 Library Imports

The next cell loads the libraries we need for this section. Don't worry about memorizing every import — just run the cell and move on. We'll explain each library as we use it.

In [ ]:
import mlx.core as mx
import mlx.nn as nn
import math
import time
from dataclasses import dataclass, field
from typing import Optional
import matplotlib.pyplot as plt
import numpy as np

# Verify MLX is available
print(f"MLX version: {mx.__version__}")
print(f"Default device: {mx.default_device()}")
print("✅ Ready to explore reasoning & test-time compute!")

---

## Visualizing the Test-Time Compute Scaling Curve

Before implementing the techniques, let's visualize the core idea: how accuracy improves as you spend more compute at inference time.

In [ ]:
# 💡 Visualize: How accuracy scales with test-time compute for different approaches

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Accuracy vs number of samples (self-consistency)
p_correct_values = [0.1, 0.3, 0.5, 0.7]
n_samples = np.arange(1, 51)

ax = axes[0]
for p in p_correct_values:
    accuracy = 1 - (1 - p) ** n_samples
    ax.plot(n_samples, accuracy, label=f"p_single = {p:.0%}", linewidth=2)

ax.set_xlabel("Number of Samples (N)", fontsize=12)
ax.set_ylabel("P(at least one correct)", fontsize=12)
ax.set_title("Self-Consistency Scaling", fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.05)

# Right: Conceptual comparison of approaches
ax = axes[1]
budget = np.linspace(1, 100, 200)

# Simulated accuracy curves for different approaches
single = np.full_like(budget, 0.4)
cot = np.full_like(budget, 0.55)
self_con = 0.55 + 0.4 * (1 - np.exp(-budget / 15))
tot = 0.55 + 0.42 * (1 - np.exp(-budget / 10))
mcts = 0.55 + 0.44 * (1 - np.exp(-budget / 8))

ax.plot(budget, single, '--', label="Standard (1-pass)", linewidth=2, color='gray')
ax.plot(budget, cot, '--', label="CoT (1-pass)", linewidth=2, color='steelblue')
ax.plot(budget, self_con, label="Self-Consistency", linewidth=2, color='green')
ax.plot(budget, tot, label="Tree-of-Thought", linewidth=2, color='orange')
ax.plot(budget, mcts, label="MCTS + PRM", linewidth=2, color='red')

ax.set_xlabel("Relative Compute Budget", fontsize=12)
ax.set_ylabel("Accuracy on Hard Problems", fontsize=12)
ax.set_title("Test-Time Compute Scaling (Conceptual)", fontsize=13)
ax.legend(fontsize=9, loc='lower right')
ax.grid(True, alpha=0.3)
ax.set_ylim(0.3, 1.05)

plt.tight_layout()
plt.savefig("data/test_time_compute_scaling.png", dpi=150, bbox_inches='tight')
plt.show()
print("⚡ Key takeaway: search-based methods (MCTS + PRM) scale best with more compute")

### 🔍 What Just Happened?

We explored the test-time compute paradigm: Chain-of-Thought prompting (free, just change the prompt), self-consistency (sample N answers, majority vote), MCTS search (systematic exploration with UCB1), and Process Reward Models (score each reasoning step). The key insight: you can trade inference compute for answer quality, and a smaller model with more search can outperform a larger model with greedy decoding.

---

## The Reasoning Landscape in 2025

Let's put the techniques in historical context and understand how the field got here.

### The Evolution of Reasoning in LLMs

| Year | Milestone | Key Idea | Impact |
|------|-----------|----------|--------|
| 2022 | **Chain-of-Thought** (Wei et al.) | "Let's think step by step" | First demonstration that prompting alone improves reasoning |
| 2022 | **Self-Consistency** (Wang et al.) | Sample N answers, majority vote | Simple but effective — 10-20% accuracy gains on math |
| 2023 | **Tree-of-Thought** (Yao et al.) | Explicit tree search over reasoning | Enabled backtracking and multi-path exploration |
| 2023 | **Let's Verify Step by Step** (Lightman et al.) | Process reward models | Dense feedback for search — score each step, not just the answer |
| 2024 | **o1** (OpenAI) | RL-trained reasoning + test-time search | First commercial "reasoning model" — dramatic gains on math/code |
| 2025 | **DeepSeek-R1** (DeepSeek) | RL for reasoning without human labels | Open-weight reasoning model, showed RL alone can teach reasoning |

💡 **The big picture:** We went from "just prompt it differently" (2022) to "train the model to reason with RL and guide it with search" (2024-2025). Each step added more compute at inference time but produced dramatically better results on hard problems.

🎯 **Interview tip:** Know the distinction between *prompting-based* reasoning (CoT, self-consistency — no model changes) and *training-based* reasoning (o1, DeepSeek-R1 — the model is trained to reason). The latter is where the field is heading, but the former is still useful and much cheaper to deploy.

⚠️ **Important note on dates:** DeepSeek-R1 was released in January 2025. Gemma 4 was released in 2025 as well. These are among the most recent developments in this rapidly evolving field.

---

## Implementation: Chain-of-Thought & Self-Consistency

💡 **What we're building:** A CoT prompting pipeline that generates step-by-step reasoning chains, and a self-consistency wrapper that samples N chains and takes the majority vote.

Since we don't have a real language model loaded, we simulate the *generation* with a simple arithmetic reasoning engine. The **algorithms** — CoT decomposition and majority voting — are exactly what production systems use.

⚡ **Key insight:** Self-consistency works because different reasoning paths may make different errors, but the correct answer tends to appear more often. It's essentially an ensemble over reasoning strategies.

In [ ]:
from utils.reasoning import ReasoningConfig, CoTPromptPipeline

# Configure the pipeline
config = ReasoningConfig(
    max_reasoning_steps=5,
    num_samples=7,       # N=7 for self-consistency
    branching_factor=3,  # For MCTS later
    temperature=0.7,     # Higher temp → more diverse (and error-prone) samples
)
print(f"ReasoningConfig: {config}")

# --- Single CoT generation ---
pipeline = CoTPromptPipeline(config)  # shape: function output
steps, answer = pipeline.generate_with_cot("What is 17 * 24?")

print("\n🔗 Chain-of-Thought reasoning:")
for s in steps:
    print(f"  {s}")
print(f"\n🎯 Final answer: {answer}")
print(f"   Correct answer: {17 * 24}")
print(f"   {'✅ Correct!' if answer == 408 else '❌ Error (expected with stochastic generation)'}")

In [ ]:
# 💡 Let's see how majority voting ACTUALLY works — no magic, just counting!
# This is the core of self-consistency (Wang et al., 2022)

from collections import Counter

# Imagine an LLM generated 7 different reasoning paths for "What is 17 × 24?"
# Each path might use a different strategy and might make different errors:
answers_from_different_paths = [408, 408, 410, 408, 406, 408, 410]

print("Self-Consistency: Majority Voting in Action")
print("=" * 50)
print(f"7 reasoning paths produced these answers: {answers_from_different_paths}\n")

# Step 1: Count how often each answer appears
vote_counts = Counter(answers_from_different_paths)
print(f"Vote counts: {dict(vote_counts)}")

# Step 2: Pick the answer with the most votes
majority_answer = vote_counts.most_common(1)[0][0]
print(f"Majority answer: {majority_answer}")
print(f"Correct answer:  {17 * 24}")
print(f"\n✅ Even though 3 out of 7 paths got it wrong, the correct answer wins!")
print(f"   This is why self-consistency works — errors are random, but the")
print(f"   correct answer tends to appear more often.")

### ⚠️ Handling Common Errors

When working with ML code, errors are normal and expected. Here's a pattern for handling them gracefully — if something goes wrong, you get a helpful message instead of a crash.

In [ ]:
# 💡 Error handling pattern — use this when operations might fail
try:
    # This is where your ML code goes
    import mlx.core as mx
    test = mx.array([1.0, 2.0, 3.0])  # shape: see output
    result = mx.sum(test)  # shape: see output
    mx.eval(result)
    print(f'✅ Success! Result: {result.item()}')
except Exception as e:
    print(f'❌ Error: {e}')
    print('💡 Tip: Check that MLX is installed and your inputs are valid')

The next cell continues the implementation:

**--- Self-Consistency: majority voting over N samples ---**

In [ ]:
# --- Self-Consistency: majority voting over N samples ---
# This is the real algorithm from Wang et al. (2022)

print("Self-Consistency with N=7 samples:\n")
majority_ans, votes, all_chains = pipeline.self_consistency(
    "What is 23 + 47?", n=7, temperature=0.8
)

for i, (chain, ans) in enumerate(zip(all_chains, [c[-1] for c in all_chains])):
    print(f"  Sample {i+1}: {chain[-1]}")

print(f"\n📊 Vote distribution: {votes}")
print(f"🎯 Majority answer: {majority_ans}")
print(f"   Correct answer: {23 + 47}")

# Demonstrate accuracy improvement with more samples
print("\n--- Accuracy vs Number of Samples ---")
import random
random.seed(42)
for n in [3, 5, 9, 15]:
    correct_count = 0
    trials = 20
    for _ in range(trials):
        ans, _, _ = pipeline.self_consistency("What is 17 * 24?", n=n, temperature=0.7)
        if ans == 408:
            correct_count += 1
    print(f"  N={n:2d} samples → accuracy: {correct_count}/{trials} = {correct_count/trials:.0%}")

print("\n💡 More samples → higher accuracy through majority voting")

---

## Implementation: MCTS Reasoning Search

🎯 **What we're building:** A Monte Carlo Tree Search (MCTS) reasoner that systematically explores reasoning paths using the same algorithm that powered AlphaGo — adapted for language model reasoning.

The four phases of MCTS:
1. **Selection** — traverse the tree using UCB1 to find a promising leaf
2. **Expansion** — generate new reasoning steps from that leaf
3. **Evaluation** — score the new steps (using a reward model or heuristic)
4. **Backpropagation** — update ancestor statistics with the evaluation result

⚡ **UCB1 formula:** $\text{UCB1}(v) = \bar{X}_v + c \sqrt{\frac{\ln N_{\text{parent}}}{N_v}}$

This balances exploitation (high-value nodes) with exploration (rarely-visited nodes). The $\sqrt{2}$ constant is provably optimal for multi-armed bandits.

⚠️ **Important:** In production systems like o1 and DeepSeek-R1, the evaluation function is a trained Process Reward Model. Here we use a heuristic evaluator for demonstration.

In [ ]:
from utils.reasoning import MCTSReasoner, ReasoningNode, ReasoningConfig

# Configure MCTS
mcts_config = ReasoningConfig(branching_factor=3, num_samples=15)
mcts = MCTSReasoner(mcts_config)  # shape: function output

# Run search
print("🌳 Running MCTS search (budget=15 iterations)...\n")
trajectory, root = mcts.search("What is 17 * 24?", budget=15)

print(f"Best trajectory: {trajectory[:120]}...")
print(f"Root visits: {root.visits}")
print(f"Root value: {root.value:.3f}")

# Show the tree structure
print("\n🌲 MCTS Tree (depth ≤ 2):")
print(mcts.tree_summary(root, max_depth=2))

The next cell continues the implementation:

**💡 Demonstrate: more budget → better exploration**

In [ ]:
# 💡 Demonstrate: more budget → better exploration

print("--- MCTS Quality vs Search Budget ---\n")
for budget in [5, 10, 20, 50]:
    mcts_run = MCTSReasoner(ReasoningConfig(branching_factor=3))
    traj, root = mcts_run.search("Solve: 15 + 28", budget=budget)
    
    # Count total nodes in tree
    def count_nodes(node):
        return 1 + sum(count_nodes(c) for c in node.children)
    
    total_nodes = count_nodes(root)
    max_depth = 0
    def get_depth(node, d=0):
        nonlocal max_depth
        max_depth = max(max_depth, d)
        for c in node.children:
            get_depth(c, d + 1)
    get_depth(root)
    
    print(f"  Budget={budget:3d} → nodes={total_nodes:4d}, "
          f"depth={max_depth}, root_value={root.value:.3f}")

print("\n⚡ More budget → deeper exploration → higher-quality reasoning paths")
print("🎯 Interview tip: MCTS + PRM is the core machinery behind o1 and DeepSeek-R1")

---

## Implementation: Process Reward Model

💡 **Process vs Outcome Reward Models — the key distinction:**

| | Outcome RM (ORM) | Process RM (PRM) |
|---|---|---|
| **What it scores** | Final answer only | Each intermediate step |
| **Feedback density** | Sparse (one signal per solution) | Dense (one signal per step) |
| **Training data** | Answer correctness labels | Step-level correctness labels |
| **Search guidance** | Can only reject bad final answers | Can prune bad reasoning paths early |
| **Cost** | Cheap to train | Expensive (need step-level annotations) |

OpenAI's "Let's Verify Step by Step" (2023) showed PRMs significantly outperform ORMs for math reasoning. The dense feedback enables much more efficient search — you don't waste compute exploring paths that went wrong in step 2.

⚠️ **Architecture note:** Our PRM uses a simple embedding + MLP architecture for educational clarity. Production PRMs are typically built on top of large pretrained transformers. The key design choice is the **sigmoid output** which guarantees scores in [0, 1].

🎯 **Interview tip:** Know that DeepSeek-R1 (2025) uses RL with process-level rewards to train reasoning capabilities directly into the model — the PRM guides the RL training signal.

In [ ]:
import mlx.core as mx
from utils.reasoning import ProcessRewardModel

# Create a small PRM for demonstration
prm = ProcessRewardModel(vocab_size=256, d_model=64, d_hidden=128, max_len=128)

# Score individual reasoning steps
print("📊 Process Reward Model — Step-Level Scoring\n")

reasoning_steps = [
    "Let x = 17 × 24",
    "17 × 24 = 17 × 20 + 17 × 4",
    "= 340 + 68",
    "= 408",
]

scores = prm.score_trajectory(reasoning_steps)
print("Step-by-step scoring (PRM):")
for step, score in zip(reasoning_steps, scores):
    bar = '█' * int(score * 30)
    print(f"  Step: {step:40s} → score: {score:.4f} {bar}")

print(f"\n✅ All scores in [0, 1]: {all(0 <= s <= 1 for s in scores)}")
print(f"   Min: {min(scores):.4f}, Max: {max(scores):.4f}")

The next cell continues the implementation:

**💡 Compare: PRM provides dense feedback vs ORM's sparse feedback**

In [ ]:
# 💡 Compare: PRM provides dense feedback vs ORM's sparse feedback

print("--- PRM vs ORM Feedback Comparison ---\n")

# Good reasoning chain
good_steps = ["Break 17×24 into parts", "17×20 = 340", "17×4 = 68", "340 + 68 = 408"]
good_scores = prm.score_trajectory(good_steps)

# Bad reasoning chain (error in step 2)
bad_steps = ["Break 17×24 into parts", "17×20 = 350", "17×4 = 68", "350 + 68 = 418"]
bad_scores = prm.score_trajectory(bad_steps)

print("Good chain (PRM scores each step):")
for step, score in zip(good_steps, good_scores):
    print(f"  {score:.3f} | {step}")

print("\nBad chain (PRM scores each step):")
for step, score in zip(bad_steps, bad_scores):
    print(f"  {score:.3f} | {step}")

print("\n💡 In a real PRM, the bad chain would show a score drop at the error step.")
print("   Our educational PRM uses random initialization, so scores reflect")
print("   architecture behavior (sigmoid bounds) rather than trained quality.")
print("\n🎯 Key takeaway: PRMs enable targeted backtracking — you know *where*")
print("   reasoning went wrong, not just *that* it went wrong.")

In [ ]:
# 💡 What would a TRAINED PRM look like? Let's simulate realistic scores.
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

print("📊 Simulated TRAINED PRM scores (what you'd see in production):\n")

# Good reasoning chain — all steps correct
good_steps_sim = ["Break 17×24 into parts", "17×20 = 340", "17×4 = 68", "340 + 68 = 408"]
good_trained_scores = [0.95, 0.92, 0.94, 0.98]

# Bad reasoning chain — error at step 2
bad_steps_sim = ["Break 17×24 into parts", "17×20 = 350 ← ERROR", "17×4 = 68", "350 + 68 = 418"]
bad_trained_scores = [0.95, 0.15, 0.12, 0.08]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.barh(range(4), good_trained_scores, color='green', alpha=0.7)
ax1.set_yticks(range(4))
ax1.set_yticklabels([s[:30] for s in good_steps_sim], fontsize=9)
ax1.set_xlim(0, 1)
ax1.set_title("✅ Correct Chain — PRM scores stay high", fontsize=11)
ax1.set_xlabel("PRM Score")

ax2.barh(range(4), bad_trained_scores, color=['green', 'red', 'red', 'red'], alpha=0.7)
ax2.set_yticks(range(4))
ax2.set_yticklabels([s[:30] for s in bad_steps_sim], fontsize=9)
ax2.set_xlim(0, 1)
ax2.set_title("❌ Error at Step 2 — PRM catches it!", fontsize=11)
ax2.set_xlabel("PRM Score")

plt.tight_layout()
plt.savefig("data/prm_comparison.png", dpi=150, bbox_inches='tight')
plt.show()
print("💡 A trained PRM's scores DROP sharply when reasoning goes wrong.")
print("   This lets MCTS prune bad paths early instead of wasting compute.")

The next cell continues the implementation:

**⚡ Integration: MCTS with Process Reward Model as evaluator**

In [ ]:
# ⚡ Integration: MCTS with Process Reward Model as evaluator

from utils.reasoning import MCTSReasoner, ProcessRewardModel, ReasoningConfig

prm_evaluator = ProcessRewardModel(vocab_size=256, d_model=64, d_hidden=128)
mcts_with_prm = MCTSReasoner(
    config=ReasoningConfig(branching_factor=3, num_samples=10),
    evaluator=prm_evaluator,  # PRM guides the search!
)

print("🌳 MCTS + PRM Integration Demo\n")
traj, root = mcts_with_prm.search("What is 15 * 12?", budget=10)

print(f"Best trajectory: {traj[:120]}")
print(f"Root visits: {root.visits}, value: {root.value:.3f}")
print("\nTree with PRM-guided evaluation:")
print(mcts_with_prm.tree_summary(root, max_depth=2))

print("\n💡 With a trained PRM, the search would strongly prefer correct")
print("   reasoning steps, making the tree much more focused.")

## 🧪 Try It Yourself

Experiment with reasoning:

1. **Self-consistency**: Generate 5 answers to a simple math problem. Do they all agree? What if you increase temperature?

2. **UCB1 exploration**: Plot the UCB1 score for a node with value=0.7 as the visit count goes from 1 to 100. When does exploration stop dominating?

3. **Process reward**: Score a 3-step reasoning chain where step 2 is wrong. How does the PRM score differ from an outcome-only reward?

---

## 📜 History & Alternatives

The evolution of reasoning in LLMs — from simple prompting tricks to sophisticated search-based systems:

| Year | Milestone | Team | Key Contribution |
|------|-----------|------|------------------|
| **2022** | **Chain-of-Thought** | Wei et al. (Google) | "Let's think step by step" — first demonstration that prompting alone improves multi-step reasoning. Zero-cost technique that became ubiquitous. |
| **2022** | **Self-Consistency** | Wang et al. (Google) | Sample N reasoning paths, majority vote on the answer. Simple but effective — 10-20% accuracy gains on math benchmarks. |
| **2023** | **Tree-of-Thought** | Yao et al. (Princeton) | Explicit tree search over reasoning states. Enabled backtracking and multi-path exploration for the first time. |
| **2023** | **Let's Verify Step by Step** | Lightman et al. (OpenAI) | Process Reward Models — score each reasoning step, not just the final answer. Dense feedback dramatically improves search efficiency. |
| **2024** | **o1** | OpenAI | First commercial "reasoning model." Combines RL-trained reasoning with test-time search. Dramatic gains on math, code, and science benchmarks. |
| **2025** | **DeepSeek-R1** | DeepSeek | Open-weight reasoning model trained with RL (GRPO) and process-level rewards. Showed that RL alone — without human reasoning labels — can teach models to reason. |

💡 **The big picture:** We went from "just prompt it differently" (2022) to "train the model to reason with RL and guide it with search" (2024–2025). Each step added more compute at inference time but produced dramatically better results on hard problems.

### The Two Paradigms

**Prompting-based reasoning** (CoT, Self-Consistency, ToT):
- No model changes needed — works with any LLM
- Cheap to deploy
- Limited by the base model's capabilities
- Still useful and widely deployed in production

**Training-based reasoning** (o1, DeepSeek-R1):
- Model is trained to reason via RL with process rewards
- Expensive to train, but produces fundamentally better reasoners
- The model generates internal reasoning chains before answering
- This is where the field is heading

⚡ **What's next?** The frontier is moving toward models that can:
- Allocate compute adaptively (easy questions → fast, hard questions → deep search)
- Learn to verify their own reasoning (self-play verification)
- Combine multiple reasoning strategies (CoT + search + verification)

🎯 **Interview tip:** Frame reasoning as a *compute allocation* problem. The key question labs are researching: for a fixed total compute budget, how much should go to training vs inference? The answer depends on the difficulty distribution of your workload.

⚠️ **Important dates:** DeepSeek-R1 was released January 2025. Gemma 4 was released in 2025. These are among the most recent developments in this rapidly evolving field.